[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_FPGA/HLS_for_FPGA.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# HLS for FPGA: C to Gates

> ⚠️ **Draft — requires an HLS toolchain (Vitis HLS, free license) not available at authoring time.** An instructor should run each block before teaching; remove this banner after.

The [FPGA workshop](./Intro_FPGA.ipynb) wrote Verilog by hand; **high-level synthesis** compiles C/C++ into it — with #pragma annotations steering the hardware. You still think in [pipelines and timing](./Intro_FPGA.ipynb); you just stop hand-placing every register. The bridge course for software engineers entering hardware.

## 1. Pre-requisites

[Intro to FPGA](./Intro_FPGA.ipynb) (what synthesis produces), [Intro to C](../Intro_Programming/Intro_C.ipynb), [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) S1 (fixed point).

---
### 🕐 Session 1 of 2 — *C With Hardware Semantics* (~40 min)
**Goal:** arbitrary-precision types, the FIR again, and the pragmas that shape silicon.
**Builds on:** [Intro to FPGA](./Intro_FPGA.ipynb) S3. &nbsp; **Feeds into:** Session 2 (interfaces & integration).

---

<details><summary>🎓 <b>Teacher notes — Session 1: C With Hardware Semantics</b></summary>

**Position this workshop precisely for the audience it's built for: software engineers who already know C, entering hardware without wanting to hand-write Verilog first.** If [Intro to FPGA](./Intro_FPGA.ipynb) hasn't been taken, at minimum walk its Session 3 (the hand-written Verilog FIR) side by side with this one — the entire value of HLS only lands once students have felt how much lower-level RTL is. This notebook is also a draft (banner at top): an HLS toolchain (Vitis HLS) is required to actually run any of this, and none of it has been synthesized by an automated check.

**The mental model shift is subtler here than in plain Verilog, and worth stating explicitly:** this *looks* like ordinary C, complete with a `for` loop and array indexing, but it still compiles to a datapath, not instructions — the loop with `#pragma HLS UNROLL` doesn't execute 4 times sequentially, it *replicates hardware* into 4 parallel multiply-accumulate units. The risk is students reading this as familiar C and missing that every pragma is silently rewriting what hardware gets generated underneath identical-looking code.

**`ap_fixed<16,1>` deserves a slow walk since the two template arguments are easy to transpose:** total bit width first, integer-bits-including-sign second — so `ap_fixed<16,1>` is 1 integer/sign bit and 15 fractional bits, i.e. exactly the Q1.15 format from the plain-Verilog FIR. This is the same fixed-point discipline from Intro to FPGA §4, just expressed as a C++ type instead of manual bit-slicing — the accumulator's extra headroom bits (`ap_fixed<34,3>`) exist for the identical reason as before: repeated multiply-accumulate needs growth room to avoid overflow.

**The synthesis report — not a green compiler checkmark — is the actual pass/fail signal, and this is worth repeating until it sticks:** II (initiation interval, samples accepted per clock) and latency come from the *HLS report*, generated after synthesis, not from whether the C++ compiles. `II=1` means one new sample per clock (the target for streaming DSP); if the report shows `II>2` or worse, something — usually a loop-carried dependency or a resource conflict — broke the pipeline, and the report itself points at where. Reading that report is the actual skill this session teaches, more than the C++ syntax is.

**The golden-model habit from Intro to FPGA returns here in a stronger form — call that out.** Because `fir()` is *ordinary C*, it compiles and runs on a laptop with no hardware involved at all: verify bit-exact agreement against a NumPy Q15 reference *before* ever touching synthesis, and let RTL cosimulation replay the same testbench against the generated hardware afterward. Same discipline as before, now with a much cheaper first checkpoint.
</details>

💡 **Intuition.** HLS C is C *reinterpreted*: loops become pipelines, arrays become BRAMs, and `#pragma` lines are floor-plan instructions. The mental shift: you are not writing instructions to execute but *describing a datapath to instantiate* — the [FPGA workshop's](./Intro_FPGA.ipynb) lesson surviving the syntax change. The report, not the compiler exit code, is the real output: II (initiation interval — samples per clock), latency, and resource counts.

```cpp
// fir.cpp — the same Q15 FIR as [Intro_FPGA §4] and [Real_Time_DSP §1], in HLS C++
#include <ap_fixed.h>
typedef ap_fixed<16, 1> q15_t;              // 1 sign+int bit, 15 fraction: hardware Q1.15
typedef ap_fixed<34, 3> acc_t;              // accumulator with headroom — the rule, again

void fir(q15_t x_in, q15_t* y_out) {
    static const q15_t H[4] = {0.1, 0.4, 0.4, 0.1};
    static q15_t delay[4];
#pragma HLS ARRAY_PARTITION variable=delay complete   // registers, not BRAM: all taps at once
    acc_t acc = 0;
tap_loop:
    for (int k = 3; k > 0; k--) {
#pragma HLS UNROLL                                     // spatial: 4 multipliers, not 1 reused
        delay[k] = delay[k-1];
        acc += delay[k] * H[k];
    }
    delay[0] = x_in;
    acc += delay[0] * H[0];
    *y_out = (q15_t)acc;
}
// synthesis target: II=1 (one sample per clock). If the report says II>1, a dependency
// or resource limit broke the pipeline — the HLS debugging loop lives in that report.
```

**The C testbench IS the golden model** (the [FPGA workshop's](./Intro_FPGA.ipynb) verification
strategy, upgraded): the same `fir()` compiles for your laptop — assert bit-exact agreement with
a NumPy Q15 reference *before* synthesis, then let cosimulation replay it against the RTL.

---
### 🕐 Session 2 of 2 — *Interfaces & Integration* (~35 min)
**Goal:** AXI-Stream in, AXI-Lite control: dropping the block into a real system.
**Builds on:** Session 1.

---

<details><summary>🎓 <b>Teacher notes — Session 2: Interfaces & Integration</b></summary>

**Frame this session as "the part that makes Session 1's filter actually usable," since a synthesizable `fir()` function alone isn't a deployable block yet.** `fir_stream` wraps the single-sample `fir()` from Session 1 in a loop that reads/writes AXI-Stream ports — this is the wiring that lets the filter sit inside a real system rather than existing only as an isolated testbench.

**Walk the two interface pragmas as answering two different questions, since conflating them is a common source of confusion:** `axis` (AXI-Stream) is the *data* path — a continuous, back-pressured stream of samples in and out, structurally identical to the producer/consumer queue pattern from the Intro to OS workshop, just implemented as a hardware handshake instead of a software queue. `s_axilite` (AXI-Lite) is the *control* path — a lightweight register interface the host CPU uses to configure the block (here, just the sample count `n`) without touching the sample stream at all. Every real accelerator needs both: a fast, narrow data channel and a slow, occasional control channel.

**`#pragma HLS PIPELINE II=1` on the streaming loop is the payoff for the discipline in Session 1** — it asks the tool to accept a new input sample every clock cycle, matching the II target from the underlying `fir()`. If this pragma fails to achieve II=1 in the report, the loop-carried dependency is almost always inside `fir()` itself (the shift-register update), which is exactly why Session 1 spent real time on getting that report-reading skill established first.

**The classic pitfall table deserves to be treated as a checklist students actually run, not prose to skim:**
1. *Unintended BRAM* — forgetting `ARRAY_PARTITION` on a small array the tool would otherwise put in slow shared memory instead of fast individual registers.
2. *II ruined by a loop-carried dependency* — a value in one iteration depends on the previous iteration's result in a way that can't be pipelined; the fix is almost always restructuring the accumulation, not fighting the tool.
3. *Float sneaking in where `ap_fixed` was meant* — an implicit conversion silently reintroduces a floating-point unit the fabric may not have, or one far more expensive than the fixed-point design intended; this is invisible in simulation and only shows up as a resource-usage surprise in the synthesis report.

**Close by connecting to real-time DSP architecture, since that's the actual destination:** DMA moving buffers without CPU involvement is the same "don't wake the CPU for every sample" principle as the Real-Time DSP workshop's architecture discussion — this streaming HLS block is the hardware half of that story, with the CPU only ever touching the AXI-Lite control registers, never the sample stream itself.
</details>

```cpp
// streaming top-level: how the block meets the [SDR](../Intro_SDR/Software_Defined_Radio.ipynb)-
// style sample flow
#include <hls_stream.h>
void fir_stream(hls::stream<q15_t>& in, hls::stream<q15_t>& out, int n) {
#pragma HLS INTERFACE axis port=in
#pragma HLS INTERFACE axis port=out
#pragma HLS INTERFACE s_axilite port=n            // CPU pokes length/config over AXI-Lite
    for (int i = 0; i < n; i++) {
#pragma HLS PIPELINE II=1
        q15_t y; fir(in.read(), &y); out.write(y);
    }
}
```

💡 **Intuition.** Interfaces are where HLS projects live or die: `axis` streams match the
[producer/consumer](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) shape of sample pipelines,
`s_axilite` gives the CPU a control panel, and DMA moves buffers without the CPU touching
samples — the [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) architecture, cast in silicon.
The classic pitfall table: unintended BRAM (missing partition pragma), II ruined by a
loop-carried dependency, and float sneaking in where `ap_fixed` was meant.

---
## Where next

- [Intro to FPGA](./Intro_FPGA.ipynb) — read the Verilog HLS emits; it demystifies both.
- [Sigma-Delta](../Intro_DSP/Sigma_Delta_Quantization.ipynb) — a decimating CIC in HLS is the perfect second project.